# HallusionBench — Kiểm chứng Báo cáo, Soi Test Case có Ảnh & Xuất Gói Model (`.pkl`)

Notebook tối giản chạy trên **Google Colab** (hoặc Jupyter local) để:
1. Kiểm tra tính toàn vẹn của kết quả thật trong `benchmarks/hallusionbench/results/`.
2. Xác minh điều kiện **0 epoch — KHÔNG train/fine-tune** từ `run_metadata.json` & `run_config.yaml`.
3. **Tái tính lại toàn bộ số liệu** trong `REPORT.md` (`aAcc`, `fAcc`, `qAcc`, `VD/VS`, `Easy/Hard`, `8 subcategories`) trực tiếp từ `predictions.jsonl`.
4. **Hiển thị từng test case kèm ảnh thật** (Text-only vs Ảnh gốc vs Ảnh bẫy chỉnh sửa) và so sánh câu trả lời của 3 mô hình.
5. Xuất gói **`<model>_bundle.pkl`** (kèm tùy chọn lưu trọng số `save_pretrained` lên Drive) để sang notebook `hallusionbench_load_and_test.ipynb` nạp lại và test sống.

## Bước 1 — Kết nối Google Drive & Chuẩn bị Repo

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/hoainhan256/vlm-hallucination-benchmark.git"
    BRANCH = "benchmark/hallusionbench"
    REPO_DIR = Path("/content/vlm-hallucination-benchmark")

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

    os.chdir(REPO_DIR)
    DRIVE_ROOT = Path("/content/drive/MyDrive/hallusionbench_runs")
    DATA_ROOT = Path("/content/data/HallusionBench")
    !pip install -q -r requirements.txt
    !pip install -q --upgrade transformers accelerate qwen-vl-utils num2words
else:
    REPO_DIR = Path.cwd().resolve()
    if not (REPO_DIR / "benchmarks" / "hallusionbench").exists():
        REPO_DIR = REPO_DIR.parents[1]
    os.chdir(REPO_DIR)
    DRIVE_ROOT = REPO_DIR / "tmp" / "hallusionbench_runs"
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", str(REPO_DIR / "tmp" / "HallusionBench")))

BUNDLE_DIR = DRIVE_ROOT / "exported_bundles"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("REPO_DIR   :", REPO_DIR)
print("DATA_ROOT  :", DATA_ROOT)
print("BUNDLE_DIR :", BUNDLE_DIR)

## Bước 2 — Kiểm tra Cấu hình Repo & Tính Toàn vẹn Artifact Báo cáo

In [ ]:
import subprocess
from vlm_benchmark.config import validate_all

print("1. Kiểm tra config chung:")
for msg in validate_all("configs/models.yaml", "configs/datasets.yaml", "configs/experiment.yaml"):
    print(" -", msg)

MODELS = ["smolvlm2", "qwen3_vl", "phi4_multimodal"]
RESULTS_ROOT = REPO_DIR / "benchmarks" / "hallusionbench" / "results"

print("\n2. Kiểm tra schema predictions.jsonl & SHA-256 manifest.json:")
for m in MODELS:
    mdir = RESULTS_ROOT / m
    r1 = subprocess.run([sys.executable, "scripts/check_results.py", str(mdir / "predictions.jsonl")], capture_output=True, text=True)
    r2 = subprocess.run([sys.executable, "scripts/check_manifest.py", str(mdir)], capture_output=True, text=True)
    print(f" - {m:16s} | check_results={r1.returncode} | check_manifest={r2.returncode}")
    if r1.returncode != 0:
        print(r1.stdout, r1.stderr)
    if r2.returncode != 0:
        print(r2.stdout, r2.stderr)

## Bước 3 — Nạp Bộ Dữ Liệu & Ảnh Thật HallusionBench

In [ ]:
import tarfile
from huggingface_hub import snapshot_download
from benchmarks.hallusionbench.dataset import HallusionBenchDataset

HF_DATASET_REPO = "rayguan/HallusionBench"
HF_DATASET_REVISION = "2067dfdce5abc7efdbb0927f4721ae282f0b6c51"
DRIVE_DATASET_TAR = DRIVE_ROOT / "dataset_cache" / f"HallusionBench_{HF_DATASET_REVISION[:8]}.tar"
DRIVE_DATASET_TAR.parent.mkdir(parents=True, exist_ok=True)

if not (DATA_ROOT / "HallusionBench.json").is_file():
    if DRIVE_DATASET_TAR.is_file():
        print("Giải nén dataset có sẵn từ Drive (không cần gọi HF):", DRIVE_DATASET_TAR)
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        with tarfile.open(DRIVE_DATASET_TAR, "r") as tf:
            tf.extractall(DATA_ROOT)
    else:
        print(f"Đang tải {HF_DATASET_REPO} @ {HF_DATASET_REVISION[:10]} vào {DATA_ROOT}...")
        os.environ["HF_HUB_DISABLE_XET"] = "1"
        hf_token = os.environ.get("HF_TOKEN")
        if not hf_token and IN_COLAB:
            try:
                from google.colab import userdata
                hf_token = userdata.get("HF_TOKEN")
            except Exception:
                hf_token = None
        snapshot_download(
            repo_id=HF_DATASET_REPO,
            repo_type="dataset",
            revision=HF_DATASET_REVISION,
            local_dir=str(DATA_ROOT),
            token=hf_token,
        )
        with tarfile.open(DRIVE_DATASET_TAR, "w") as tf:
            for item in DATA_ROOT.iterdir():
                if item.name != ".cache":
                    tf.add(item, arcname=item.name)

dataset = HallusionBenchDataset(
    json_path=DATA_ROOT / "HallusionBench.json",
    data_root=DATA_ROOT,
    dataset_source=HF_DATASET_REPO,
    dataset_revision=HF_DATASET_REVISION,
)
dataset_index = dataset.load_index(verify_images=True, verify_image_readable=False)
print(f"Đã nạp {len(dataset_index)} câu hỏi thật từ HallusionBench.json")

## Bước 4 — Xác minh "0 Epoch (Không Train)" & Tái tính Số liệu `REPORT.md`

In [ ]:
import json
import yaml
from statistics import mean
from benchmarks.hallusionbench.evaluation import score_hallusionbench
from vlm_benchmark.results import read_jsonl

print("=== 1. XÁC MINH THIẾT LẬP ZERO-SHOT (0 EPOCH — KHÔNG TRAIN) ===")
for m in MODELS:
    meta = json.loads((RESULTS_ROOT / m / "run_metadata.json").read_text(encoding="utf-8"))
    cfg = yaml.safe_load((RESULTS_ROOT / m / "run_config.yaml").read_text(encoding="utf-8"))
    print(
        f"- {m:16s} | rev={meta['resolved_model_revision'][:10]} | "
        f"dtype={cfg.get('dtype')} | do_sample={cfg.get('generation_config', {}).get('do_sample')} | "
        f"gpu={meta.get('environment', {}).get('gpu')} | stage={meta.get('stage')}"
    )

print("\n=== 2. TÁI TÍNH TRỰC TIẾP TỪ predictions.jsonl & ĐỐI CHIẾU REPORT.md ===")
all_preds = {}
rows = []
for m in MODELS:
    preds = read_jsonl(RESULTS_ROOT / m / "predictions.jsonl")
    all_preds[m] = {p["sample_id"]: p for p in preds}
    recomputed, errs = score_hallusionbench(preds, dataset_index)
    saved = json.loads((RESULTS_ROOT / m / "metrics.json").read_text(encoding="utf-8"))

    assert recomputed["metrics"] == saved["metrics"], f"Lệch metrics ở {m}!"
    avg_lat_s = mean(p["timing_ms"] for p in preds) / 1000.0
    met = recomputed["metrics"]
    cat = recomputed["by_category"]
    diff = recomputed["by_difficulty"]
    rows.append(
        f"{m:16s} | {len(preds)}/34 | AvgLat={avg_lat_s:5.2f}s | "
        f"aAcc={met['aAcc']*100:5.2f}% | fAcc={met['fAcc']*100:5.2f}% | qAcc={met['qAcc']*100:5.2f}% | "
        f"VD={cat['VD']['aAcc']*100:5.2f}% | VS={cat['VS']['aAcc']*100:5.2f}% | "
        f"Easy={diff['easy']['aAcc']*100:5.2f}% | Hard={diff['hard']['aAcc']*100:5.2f}%"
    )

print("\n".join(rows))
print("\n--> KHỚP 100% với metrics.json và bảng điểm trong REPORT.md!")

## Bước 5 — Hiển thị 5 Biểu đồ của Báo cáo (`REPORT.md`)

In [ ]:
from IPython.display import Image as IPyImage, display

fig_dir = RESULTS_ROOT / "figures"
for fig_path in sorted(fig_dir.glob("*.png")):
    print(f"\n--- {fig_path.name} ---")
    display(IPyImage(filename=str(fig_path), width=850))

## Bước 6 — Soi Từng Test Case Kèm Ảnh Thật (Ảnh gốc vs Ảnh bẫy vs Không ảnh)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from benchmarks.hallusionbench.evaluation import parse_yes_no

pilot_sample_ids = list(all_preds["smolvlm2"].keys())
disagreements = json.loads((RESULTS_ROOT / "cross_model_disagreements.json").read_text(encoding="utf-8"))
print(f"Tổng số câu pilot: {len(pilot_sample_ids)} | Số ca 3 model bất đồng: {len(disagreements)}")

def inspect_group(group_key: str) -> None:
    """Hiển thị tất cả biến thể (text-only / ảnh gốc / ảnh bẫy) trong cùng 1 nhóm đối chứng."""
    members = [
        dataset_index[sid] for sid in pilot_sample_ids
        if dataset_index[sid]["group_or_pair_key"] == group_key
    ]
    if not members:
        print("Không tìm thấy nhóm:", group_key)
        return

    print("=" * 80)
    print(f"NHÓM ĐỐI CHỨNG: {group_key} ({len(members)} câu hỏi)")

    # Hiển thị các ảnh duy nhất trong nhóm
    unique_imgs = []
    seen_paths = set()
    for rec in members:
        p = rec.get("image_path")
        if p and p not in seen_paths:
            seen_paths.add(p)
            unique_imgs.append((rec["visual_input_raw"], rec["figure_id"], p))

    if unique_imgs:
        fig, axes = plt.subplots(1, len(unique_imgs), figsize=(6 * len(unique_imgs), 4))
        if len(unique_imgs) == 1:
            axes = [axes]
        for ax, (vis_raw, fig_id, img_path) in zip(axes, unique_imgs):
            label = "Ảnh GỐC (vis=1)" if vis_raw == "1" else "Ảnh BẪY/CHỈNH SỬA (vis=2)"
            ax.imshow(Image.open(img_path).convert("RGB"))
            ax.set_title(f"fig_{fig_id} — {label}")
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    # In từng câu hỏi & kết quả 3 model
    for rec in members:
        sid = rec["sample_id"]
        gt = rec["reference_answer"]
        print(f"\n[{sid}] (visual_input={rec['visual_input_raw']}, difficulty={rec.get('difficulty')})")
        print(f"  Q : {rec['question']}")
        print(f"  GT: {gt} ({rec.get('gt_answer_details', '')})")
        for m in MODELS:
            raw = all_preds[m][sid]["raw_prediction"]
            parsed = parse_yes_no(raw)
            ok = "ĐÚNG" if parsed == gt else "SAI "
            print(f"  - {m:16s}: [{ok}] parsed={parsed} | raw={raw!r}")

# Liệt kê nhanh các nhóm có trong pilot:
groups = sorted({dataset_index[sid]["group_or_pair_key"] for sid in pilot_sample_ids})
print("Các nhóm đối chứng trong pilot:", groups)

# Ví dụ soi 2 nhóm điển hình (bạn có thể đổi sang bất kỳ nhóm nào ở trên):
inspect_group("VS::table::set_0")
inspect_group("VD::illusion::set_12")

## Bước 7 — Xuất Gói Model (`.pkl` Bundle) sau khi Chạy xong để Test ở Notebook Riêng

Cell này nạp model zero-shot (0 epoch) và lưu file **`<MODEL_KEY>_bundle.pkl`** (chứa metadata, `HallusionRunConfig`, `prompt_config`, đường dẫn trọng số `save_pretrained` trên Drive nếu bật `SAVE_WEIGHTS_TO_DRIVE = True`) để bạn nạp lại ở file **`hallusionbench_load_and_test.ipynb`**.

> Lưu ý: Nếu chạy `phi4_multimodal` trên Colab, cần `transformers==4.48.2` (`!pip install -q transformers==4.48.2 accelerate==1.3.0 peft==0.13.2`).

In [ ]:
import pickle
import torch
from benchmarks.hallusionbench.inference import (
    HallusionRunConfig,
    HuggingFaceHallusionModel,
    MODEL_SPECS,
)

MODEL_KEY = "smolvlm2"          # Chọn: "smolvlm2" | "qwen3_vl" | "phi4_multimodal"
SAVE_WEIGHTS_TO_DRIVE = False   # Đổi thành True nếu muốn lưu cả trọng số save_pretrained() lên Drive (~4-8GB)

meta = json.loads((RESULTS_ROOT / MODEL_KEY / "run_metadata.json").read_text(encoding="utf-8"))
dtype = "float16" if torch.cuda.is_available() else "float32"
attn_impl = "eager" if MODEL_KEY == "phi4_multimodal" else "sdpa"

run_cfg = HallusionRunConfig(
    experiment_id="hb_export_bundle",
    run_id=f"export_{MODEL_KEY}",
    model_key=MODEL_KEY,
    model_id=MODEL_SPECS[MODEL_KEY]["model_id"],
    stage="pilot",
    model_revision=meta["resolved_model_revision"],
    processor_revision=meta["resolved_processor_revision"],
    dtype=dtype,
    attn_implementation=attn_impl,
    device_map="auto" if torch.cuda.is_available() else "cpu",
    trust_remote_code=MODEL_SPECS[MODEL_KEY]["default_trust_remote_code"],
    dataset_revision=HF_DATASET_REVISION,
)

saved_weights_dir = None
if SAVE_WEIGHTS_TO_DRIVE:
    print(f"Đang nạp {MODEL_KEY} để lưu trọng số vào Drive...")
    adapter = HuggingFaceHallusionModel(run_cfg)
    adapter.load_model()
    saved_weights_dir = str(BUNDLE_DIR / f"{MODEL_KEY}_weights")
    adapter.model.save_pretrained(saved_weights_dir)
    adapter.processor.save_pretrained(saved_weights_dir)
    adapter.release_resources()

bundle = {
    "model_key": MODEL_KEY,
    "model_id": run_cfg.model_id,
    "model_revision": run_cfg.model_revision,
    "processor_revision": run_cfg.processor_revision,
    "dtype": dtype,
    "attn_implementation": attn_impl,
    "trust_remote_code": run_cfg.trust_remote_code,
    "prompt_config": run_cfg.prompt_config.to_dict(),
    "generation_config": run_cfg.generation_config_dict(),
    "saved_weights_dir": saved_weights_dir,
    "report_metrics": json.loads((RESULTS_ROOT / MODEL_KEY / "metrics.json").read_text(encoding="utf-8"))["metrics"],
}

pkl_path = BUNDLE_DIR / f"{MODEL_KEY}_bundle.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(bundle, f)

print("Đã xuất thành công:", pkl_path)
print("Nội dung bundle:", {k: v for k, v in bundle.items() if k != "prompt_config"})